In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
!pip install -q datasets

In [4]:
from datasets import load_dataset, DatasetDict

ds = load_dataset("b-mc2/sql-create-context", split = "train")
ds = ds.shuffle(seed = 42).select(range(3000))

s1 = ds.train_test_split(test_size = 0.2, seed = 42)
s2 = s1["test"].train_test_split(test_size = 0.5, seed = 42)

data = DatasetDict ({
    "train": s1["train"],
    "validation": s2["train"],
    "test": s2["test"]
})
print({k: len(v) for k, v in data.items()})
print(data["train"][0])

{'train': 2400, 'validation': 300, 'test': 300}
{'answer': 'SELECT customer_details FROM Customers EXCEPT SELECT T1.customer_details FROM Customers AS T1 JOIN Customer_Policies AS T2 ON T1.customer_id = T2.customer_id', 'question': 'List the details of the customers who do not have any policies.', 'context': 'CREATE TABLE Customer_Policies (customer_id VARCHAR); CREATE TABLE Customers (customer_details VARCHAR, customer_id VARCHAR); CREATE TABLE Customers (customer_details VARCHAR)'}


In [9]:
def build_prompt(ex):
    return (
        f"### Schema:\n{ex['context']}\n"
        f"### Question:\n{ex['question']}\n"
        f"### SQL:\n"
    )

def build_full(ex):
    return build_prompt(ex) + ex["answer"]

print(build_full(data["train"][0]))

### Schema:
CREATE TABLE Customer_Policies (customer_id VARCHAR); CREATE TABLE Customers (customer_details VARCHAR, customer_id VARCHAR); CREATE TABLE Customers (customer_details VARCHAR)
### Question:
List the details of the customers who do not have any policies.
### SQL:
SELECT customer_details FROM Customers EXCEPT SELECT T1.customer_details FROM Customers AS T1 JOIN Customer_Policies AS T2 ON T1.customer_id = T2.customer_id


In [10]:
data.save_to_disk("/kaggle/working/sql_data")

Saving the dataset (0/1 shards):   0%|          | 0/2400 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/300 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/300 [00:00<?, ? examples/s]